
# AI-Based Crop Health Monitoring Using Drone Multispectral Data

## End-to-End AI/ML Capstone Project

### Dataset Source
Google Sheets Dataset:
https://docs.google.com/spreadsheets/d/1wPL7_G65NBY7801PfKhbsM7ujANoID6DIzb2zmcJ1yM/edit?usp=sharing

---

## Objectives
- Understand vegetation indices
- Build a crop stress prediction model
- Evaluate ML model performance
- Generate spatial stress visualizations
- Recommend drone inspection strategies
- Interpret agronomy insights


In [ ]:

# Install dependencies
!pip install pandas numpy matplotlib seaborn scikit-learn plotly gdown --quiet


In [ ]:

# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings('ignore')



# Step 1: Load Dataset from Google Sheets


In [ ]:

# Google Sheet CSV export URL

sheet_id = "1wPL7_G65NBY7801PfKhbsM7ujANoID6DIzb2zmcJ1yM"
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(csv_url)

df.head()


In [ ]:

# Dataset overview

print("Dataset Shape:", df.shape)
print("\nColumns:\n", df.columns)

df.info()



# Step 2: Understanding Vegetation Indices

### Key Indices

- NDVI → Measures vegetation greenness and chlorophyll activity
- GNDVI → Sensitive to nitrogen concentration and photosynthesis
- SAVI → Reduces soil brightness effects
- EVI → Improves monitoring in dense vegetation
- Moisture Index → Detects water stress
- Canopy Density → Indicates crop coverage


In [ ]:

# Missing values

df.isnull().sum()



# Step 3: Exploratory Data Analysis


In [ ]:

# Correlation Heatmap

plt.figure(figsize=(12,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='Greens')
plt.title('Feature Correlation Heatmap')
plt.show()


In [ ]:

# Feature distributions

df.hist(figsize=(14,10), bins=20)
plt.tight_layout()
plt.show()



# Step 4: Data Preprocessing


In [ ]:

# Replace target column name if needed

target_column = 'Crop_Stress'

encoder = LabelEncoder()
df[target_column] = encoder.fit_transform(df[target_column])

X = df.drop(columns=[target_column])
y = df[target_column]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)



# Step 5: Train Machine Learning Model


In [ ]:

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Model Training Completed")



# Step 6: Model Evaluation


In [ ]:

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Precision : {precision:.2f}")
print(f"Recall    : {recall:.2f}")
print(f"F1 Score  : {f1:.2f}")
print(f"ROC-AUC   : {roc_auc:.2f}")


In [ ]:

# Classification report

print(classification_report(y_test, y_pred))


In [ ]:

# Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:

# ROC Curve

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')

plt.show()



# Step 7: Feature Importance


In [ ]:

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance


In [ ]:

sns.barplot(
    data=importance,
    x='Importance',
    y='Feature'
)

plt.title('Feature Importance')
plt.show()



# Step 8: Spatial Stress Visualization


In [ ]:

# Ensure dataset has Latitude and Longitude columns

df['Predicted_Stress'] = model.predict(X_scaled)

fig = px.scatter_mapbox(
    df,
    lat='Latitude',
    lon='Longitude',
    color='Predicted_Stress',
    hover_data=['NDVI'],
    zoom=10,
    height=600,
    title='Crop Stress Heatmap'
)

fig.update_layout(mapbox_style='open-street-map')

fig.show()



# Step 9: Drone Inspection Recommendations


In [ ]:

def recommendation(stress):
    if stress == 1:
        return 'Immediate Drone Inspection Recommended'
    return 'Routine Monitoring'

df['Recommendation'] = df['Predicted_Stress'].apply(recommendation)

df[['Predicted_Stress', 'Recommendation']].head()



# Step 10: Reflection and Improvements

## Limitations
- Weather and atmospheric conditions may affect vegetation indices
- Real-world drone imagery introduces noise
- Model performance depends on dataset quality

## Future Improvements
- Use deep learning models
- Integrate weather and soil datasets
- Deploy using Flask/FastAPI
- Add real-time drone analytics



# Conclusion

This project demonstrates a complete AI-powered crop health monitoring workflow using multispectral vegetation indices and machine learning.

The project includes:
- Data preprocessing
- ML model training
- Model evaluation
- Spatial visualization
- Drone-based agronomy insights

Suitable for AI/ML and Data Science portfolios.
